In [ ]:
# Setup: garante que .env seja carregado e pacote esteja no path
import sys, os
from pathlib import Path

# Localiza raiz do projeto (sobe até encontrar pyproject.toml)
project_root = Path(os.getcwd())
for p in [project_root, *project_root.parents]:
    if (p / 'pyproject.toml').exists():
        project_root = p
        break

# Adiciona src/ ao sys.path (caso o pacote não esteja instalado em editable mode)
src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Carrega .env da raiz do projeto
from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

print(f'Project root: {project_root}')
print(f'FRED_API_KEY: {"SET" if os.getenv("FRED_API_KEY") else "NOT SET — verifique o .env"}')

# Analise de Renda Fixa — Tesouro Direto

Analise de titulos publicos, curva de juros reais e comparativo com benchmarks.

Fontes:
- **TesouroDiretoFetcher**: taxas atuais, historico NTN-B, curva de juros reais
- **BCBFetcher**: Selic, CDI, IPCA (contexto para renda fixa)

**Nota**: Nao requer API keys. Dados publicos do Tesouro Nacional e BCB.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Taxas Atuais — Tesouro Direto

In [ ]:
from carteira_auto.data.fetchers import TesouroDiretoFetcher

tesouro = TesouroDiretoFetcher()

# Taxas atuais de todos os titulos
taxas = tesouro.get_current_rates()
print(f"Titulos disponiveis: {len(taxas)}")
print()
print(taxas.to_string(index=False))

In [ ]:
# Visualizar taxas por tipo de título
if not taxas.empty and "taxa_rendimento" in taxas.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Ordenar por taxa
    taxas_sorted = taxas.sort_values("taxa_rendimento", ascending=True)
    cores_titulo = []
    for nome in taxas_sorted["nome"]:
        if "IPCA" in str(nome):
            cores_titulo.append("#2ca02c")
        elif "Selic" in str(nome):
            cores_titulo.append("#1f77b4")
        else:
            cores_titulo.append("#ff7f0e")
    
    ax.barh(range(len(taxas_sorted)), taxas_sorted["taxa_rendimento"],
            color=cores_titulo, alpha=0.85, height=0.6)
    ax.set_yticks(range(len(taxas_sorted)))
    ax.set_yticklabels(taxas_sorted["nome"], fontsize=8)
    ax.set_title("Taxas de Rendimento — Tesouro Direto")
    ax.set_xlabel("Taxa (% a.a.)")
    plt.tight_layout()
    plt.show()
else:
    print("Coluna 'taxa_rendimento' não disponível — verifique formato do TesouroDiretoFetcher")

## 2. Curva NTN-B — Juros Reais

A curva NTN-B mostra as taxas reais (acima do IPCA) para diferentes vencimentos.
Util para avaliar o premio de risco e a expectativa de inflacao.

In [ ]:
# Curva NTN-B (juros reais por vencimento)
curva = tesouro.get_ntnb_curve()
print("Curva NTN-B (juros reais):")
print(curva)

if not curva.empty:
    fig, ax = plt.subplots()
    ax.plot(curva.index, curva.values, "o-", markersize=8, linewidth=2, color="#d62728")
    ax.set_title("Curva de Juros Reais (NTN-B)")
    ax.set_xlabel("Vencimento")
    ax.set_ylabel("Taxa real (% a.a.)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3. Historico NTN-B Principal

In [ ]:
# Historico de precos da NTN-B Principal
ntnb = tesouro.get_ntnb_history(com_cupom=False)
print(f"NTN-B Principal historico: {len(ntnb)} registros")
if not ntnb.empty:
    print(ntnb.tail(5))

## 4. Contexto — Selic, CDI e IPCA

Para avaliar renda fixa, comparamos com os benchmarks de referencia.

In [ ]:
from carteira_auto.data.fetchers import BCBFetcher

bcb = BCBFetcher()

selic = bcb.get_selic(period_days=30)
ipca = bcb.get_ipca(period_days=365)

selic_atual = selic["valor"].iloc[-1]
ipca_12m = ((1 + ipca.tail(12)["valor"] / 100).prod() - 1) * 100

print(f"Selic meta: {selic_atual:.2f}% a.a.")
print(f"IPCA acumulado 12m: {ipca_12m:.2f}%")
print(f"Juro real (Selic - IPCA): {selic_atual - ipca_12m:.2f}% a.a.")

In [ ]:
# Comparativo visual de rentabilidades
fig, ax = plt.subplots(figsize=(8, 5))

labels_rf = ["Selic nominal", "Selic real", "Poupança\n(≈70% Selic)"]
valores_rf = [selic_atual, selic_atual - ipca_12m, selic_atual * 0.7]

if not curva.empty:
    labels_rf.append(f"NTN-B longa\n(real)")
    valores_rf.append(float(curva.iloc[-1]))

cores_rf = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd"][:len(labels_rf)]

bars = ax.bar(labels_rf, valores_rf, color=cores_rf, alpha=0.85, width=0.5)
for bar, val in zip(bars, valores_rf, strict=False):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=11)

ax.set_title("Comparativo de Rentabilidades — Renda Fixa")
ax.set_ylabel("% a.a.")
ax.axhline(y=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Comparativo: NTN-B vs Selic real
print("\n=== Comparativo de Rentabilidade ===")
print(f"Selic nominal: {selic_atual:.2f}% a.a.")
print(f"Selic real (- IPCA): {selic_atual - ipca_12m:.2f}% a.a.")
if not curva.empty:
    ntnb_longa = curva.iloc[-1]
    print(f"NTN-B longa (juro real): {ntnb_longa:.2f}% a.a.")
    print(f"\nNTN-B longa vs Selic real: {ntnb_longa - (selic_atual - ipca_12m):+.2f} pp")

## Resumo

| Titulo | Tipo | Referencia | Melhor para |
|--------|------|------------|-------------|
| Tesouro Selic | Pos-fixado | Selic | Liquidez, reserva de emergencia |
| Tesouro IPCA+ | Inflacao + real | IPCA + taxa | Protecao inflacionaria, longo prazo |
| Tesouro Prefixado | Taxa fixa | CDI | Cenario de queda de juros |